In [49]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader , Dataset
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [50]:
torch.manual_seed(42)

In [51]:
df = pd.read_csv("fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [52]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [53]:
X_train , X_test , y_train , y_test = train_test_split(X, y , test_size=0.2 , random_state=42)

In [54]:
# Scaling 
X_train = X_train/255.0
X_test = X_test/255.0

In [55]:
class CustomDataset(Dataset):
    def __init__(self  , features , labels):
        
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index] , self.labels[index]        

In [56]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test, y_test)

In [57]:
train_loader = DataLoader(train_dataset,batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32, shuffle=False)

In [58]:
class MYNN(nn.Module):
    
    def __init__(self , num_features):
        
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.ReLU(),
            nn.Linear(128 , 64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    
    def forward(self , X):
        return self.model(X)
            

In [65]:
learning_rate = 0.1
epochs = 100


In [66]:
model = MYNN(X_train.shape[1])

loss_ = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(),lr=learning_rate)


for epoch in range(epochs):
    total_epoch_loss = 0
    
    for batch_features , batch_labels in train_loader:
        
        outputs = model(batch_features)
        
        loss = loss_(outputs, batch_labels)
        
        optimizer.zero_grad()
        
        loss.backward()
        
        optimizer.step()
        
        total_epoch_loss += loss.item()
    
    avg_loss = total_epoch_loss/len(train_loader)
    print(f"Epoch: {epoch + 1}, Loss:{avg_loss}")
        

Epoch: 1, Loss:1.3661302661895751
Epoch: 2, Loss:0.7981703887383144
Epoch: 3, Loss:0.6577216243743896
Epoch: 4, Loss:0.594638316432635
Epoch: 5, Loss:0.5507753390073776
Epoch: 6, Loss:0.5182666998108229
Epoch: 7, Loss:0.4913369955619176
Epoch: 8, Loss:0.4538464644054572
Epoch: 9, Loss:0.4241745497783025
Epoch: 10, Loss:0.40793799072504044
Epoch: 11, Loss:0.3950086063146591
Epoch: 12, Loss:0.37955526888370517
Epoch: 13, Loss:0.3598324364920457
Epoch: 14, Loss:0.33647893180449806
Epoch: 15, Loss:0.33211303984125456
Epoch: 16, Loss:0.3107455933094025
Epoch: 17, Loss:0.3003944789369901
Epoch: 18, Loss:0.2924530748526255
Epoch: 19, Loss:0.281188262651364
Epoch: 20, Loss:0.2697919268409411
Epoch: 21, Loss:0.27207332144180935
Epoch: 22, Loss:0.2497503490249316
Epoch: 23, Loss:0.23611941533784073
Epoch: 24, Loss:0.23932108640670777
Epoch: 25, Loss:0.22592610108355682
Epoch: 26, Loss:0.22390660824875036
Epoch: 27, Loss:0.21216911738117536
Epoch: 28, Loss:0.20796879986921946
Epoch: 29, Loss:0.19

In [67]:
batch_features, batch_labels = next(iter(train_loader))

outputs = model(batch_features)

print("batch_features:", batch_features.shape)
print("batch_labels:", batch_labels.shape)
print("batch_labels dtype:", batch_labels.dtype)
print("outputs:", outputs.shape)
print("batch_labels:", batch_labels[:5])

batch_features: torch.Size([32, 784])
batch_labels: torch.Size([32])
batch_labels dtype: torch.int64
outputs: torch.Size([32, 10])
batch_labels: tensor([1, 1, 5, 3, 0])


In [68]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        outputs = model(batch_features)

        predictions = torch.argmax(outputs, dim=1)

        total += batch_labels.size(0)
        correct += (predictions == batch_labels).sum().item()

print("Test Accuracy:", correct / total)

Test Accuracy: 0.8358333333333333
